# Pandas' Function application

**Estimated Time**: 25 minutes.

In [1]:
import pandas as pd
import numpy as np

Functions can be applied:

- to each row or column (row/column-wise)
- to each individual element (element-wise)

Or it may be applied to the data structure as a whole, whether `DataFrame` or `Series`.

### 1. Tablewise Function Application: [`pipe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html#pandas.DataFrame.pipe)

A pipeline is a sequence of data processing steps. In Pandas:

- input: `DataFrame`, output: `DataFrame`
- input: `Series`, output: `Series`

Example:

In [3]:
def extract_city_name(df):
    """
    Chicago, IL -> Chicago for city_name column
    """
    df["city_name"] = df["city_and_code"].str.split(",").str.get(0)
    return df

def add_country_name(df, country_name=None):
    """
    Chicago -> Chicago-US for city_name column
    """
    col = "city_name"
    df["city_and_country"] = df[col] + country_name
    return df

df_p = pd.DataFrame({"city_and_code": ["Chicago, IL"]})
df_p

,city_and_code
0,"Chicago, IL"


Regular function calling works, but it can get messy:

In [ ]:
add_country_name(extract_city_name(df_p), country_name="US")

,city_and_code,city_name,city_and_country
0,"Chicago, IL",Chicago,ChicagoUS


The issue with nested function calls is that they can be hard to read. For example, consider the following code (5 steps):

```python
result = format_currency(
    calculate_tax(
        add_country_name(
            extract_city_name(
                clean_date_format(df_p)
            ), 
            country_name="US"
        ), 
        tax_rate=0.08
    )
)
```

The issue with this **inside-out** is:

- Can you easily see the order of operations? (`clean_date_format -> extract_city_name -> ...`)
- Can you easily add or remove steps?
- Can you easily disable just one step for debugging?
- Can you easily tell which parameters go with which function?

Using the `pipe()` you can keep the logic linear (top-to-bottom) and improve readability:

```python
result = (df_p
    .pipe(clean_date_format)
    .pipe(extract_city_name)
    .pipe(add_country_name, country_name="US")
    .pipe(calculate_tax, tax_rate=0.08)
    .pipe(format_currency)
)
```
> Note: The parentheses around the entire expression are necessary to allow line breaks.

Advantages of using `pipe()`:

- The order is clear (top-to-bottom).
- You can easily add/remove steps.
- You can easily disable just one step for debugging.
- You can easily tell which parameters go with which function.

Let's apply that to our example functions:

In [ ]:
(
    df_p
    .pipe(extract_city_name)
    .pipe(add_country_name, country_name="US")
)

,city_and_code,city_name,city_and_country
0,"Chicago, IL",Chicago,ChicagoUS


### 2. Row or Column-wise Function Application: [`apply(func, axis=0)`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html#pandas.DataFrame.apply)

`axis`: Axis along which the function is applied:

- `0` or `'index'`: apply function to each column.
- `1` or `'columns'`: apply function to each row.

let’s create some example objects:

In [4]:
index = pd.date_range("1/1/2000", periods=8)
s = pd.Series(np.random.randn(5), index=["a", "b", "c", "d", "e"])
df = pd.DataFrame(np.random.randn(8, 3), index=index, columns=["A", "B", "C"])

In [7]:
df

,A,B,C
2000-01-01,-1.746626,-0.077106,0.564823
2000-01-02,0.671546,-0.926937,-0.641332
2000-01-03,1.524287,0.609175,0.858964
2000-01-04,0.456853,0.966903,1.109331
2000-01-05,-0.814516,0.663351,0.820062
2000-01-06,-0.411261,-0.324345,-0.081697
2000-01-07,-1.044205,0.026590,-0.795514
2000-01-08,-0.094298,-0.540062,-0.076644


In [ ]:
def absolute_mean(x: pd.Series):
    return x.abs().mean() ** 2

In [ ]:
df.apply(absolute_mean, axis=0) # column-wise mean

A    0.033225
B    0.002470
C    0.048290
dtype: float64

In [ ]:
df.apply(absolute_mean, axis=1) # row-wise mean

2000-01-01    0.176095
2000-01-02    0.089346
2000-01-03    0.994957
2000-01-04    0.712947
2000-01-05    0.049714
2000-01-06    0.074220
2000-01-07    0.365271
2000-01-08    0.056170
Freq: D, dtype: float64

Using a broadcastable operation (like addition, multiplication, etc.) means the axis parameter doesn't matter:

- Example: `axis=0` (column-wise) or `axis=1` (row-wise) when doing `x + 10`

In [14]:
def my_square(x: pd.Series):
    return x ** 2

# Equal results whether applied column-wise or row-wise
df.apply(my_square, axis=0)

,A,B,C
2000-01-01,3.050703,0.005945,0.319025
2000-01-02,0.450973,0.859213,0.411306
2000-01-03,2.323449,0.371094,0.737819
2000-01-04,0.208714,0.934901,1.230614
2000-01-05,0.663436,0.440034,0.672501
2000-01-06,0.169136,0.105199,0.006674
2000-01-07,1.090365,0.000707,0.632842
2000-01-08,0.008892,0.291667,0.005874


In [18]:
# Using lambda functions
df.apply(lambda x: x ** 2, axis=1) # row-wise mean

,A,B,C
2000-01-01,3.050703,0.005945,0.319025
2000-01-02,0.450973,0.859213,0.411306
2000-01-03,2.323449,0.371094,0.737819
2000-01-04,0.208714,0.934901,1.230614
2000-01-05,0.663436,0.440034,0.672501
2000-01-06,0.169136,0.105199,0.006674
2000-01-07,1.090365,0.000707,0.632842
2000-01-08,0.008892,0.291667,0.005874


### 3. Applying Elementwise Functions: [`map()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html#pandas.DataFrame.map)

The method `map()`:

- Accepts any Python function taking a single value and returning a single value
- Works for both `Series` and `DataFrame`

For example:

In [22]:
def f(x: pd.DataFrame | pd.Series):
    return x + 10

In [23]:
df.map(f)

,A,B,C
2000-01-01,8.253374,9.922894,10.564823
2000-01-02,10.671546,9.073063,9.358668
2000-01-03,11.524287,10.609175,10.858964
2000-01-04,10.456853,10.966903,11.109331
2000-01-05,9.185484,10.663351,10.820062
2000-01-06,9.588739,9.675655,9.918303
2000-01-07,8.955795,10.026590,9.204486
2000-01-08,9.905702,9.459938,9.923356


In [24]:
df['A'].map(f)

2000-01-01     8.253374
2000-01-02    10.671546
2000-01-03    11.524287
2000-01-04    10.456853
2000-01-05     9.185484
2000-01-06     9.588739
2000-01-07     8.955795
2000-01-08     9.905702
Freq: D, Name: A, dtype: float64

In [25]:
df.loc['2000-01-04'].map(f)

A    10.456853
B    10.966903
C    11.109331
Name: 2000-01-04 00:00:00, dtype: float64

---

References:

- [Pandas' User Guide > Essential basic functionality > Function application](https://pandas.pydata.org/docs/user_guide/basics.html#function-application)